# Week 2 Day 3: Web UI for AI Brochure Generator
We will build a browser-based application where a user can enter a company website URL, select a local model, and see a professional sales brochure generated and streamed in real-time.

## Features:
1. **URL Input:** Allows users to scrape any static homepage.
2. **Model Dropdown:** Selects either `llama3.2:1b` or `gemma3:4b`.
3. **Streaming Markdown Output:** Displays the brochure copy page-by-page as it is written.

In [20]:
import os
import json
import gradio as gr
from openai import OpenAI

# Import the scraping functions we wrote in Week 1
from scraper import fetch_website_content,fetch_website_links

print("Imports successful! Custom scraper loaded.")

Imports successful! Custom scraper loaded.


In [21]:
# Connect to our local Ollama server using the standard OpenAI client
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("Ollama connection active.")

Ollama connection active.


### Function 1: Link Filterer

In [22]:
from urllib import response


def filter_relevant_links(url, model_selection):
    """
    Scrapes the homepage, passes links to the LLM,
    and returns a list of URLs relevant to a brochure.
    """

    # 1. Fetch all raw links using our scraper.py function
    raw_links = fetch_website_links(url)

    if not raw_links:
        print("No links found on this website.")
        return []

    # 2. Set up system prompt for filtering
    link_system_prompt = """
    You are a web crawler selector. Decide which links on a homepage are relevant for a company brochure (About, Careers, Services).
    Do not include Terms of Service, Privacy Policies, or social media links.
    You MUST respond in clean JSON format matching this structure:
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.url/careers"}
        ]
    }
    """

    # 3. create the User prompt
    user_prompt = f"Target Website: {url}\nLinks:\n" + "\n".join(raw_links)

    # 4. Call the local model forcing JSON formate
    response = client.chat.completions.create(
        model=model_selection,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
    )

    # 5. Parse and return the list of selected link object
    raw_json = response.choices[0].message.content
    data = json.loads(raw_json)


    # Return the 'links' list from our JSON (or an empty list if missing)
    return data.get("links",[])

### Function 2: Content Scraper

In [23]:
def scrape_all_data(homepage_url, selected_links):
    """
    Scrapes the contents of the homepage and all selected sub-pages, 
    combining them into one large text block.
    """

    combined_text = ""

    # 1. Scrape the main homepage first
    print(f"Scraping homepage: {homepage_url}")
    combined_text += f"=== HOMEPAGE CONTENT ===\n"
    combined_text += fetch_website_content(homepage_url) + "\n\n"

    # 2. Scrap each of the relevant sub-pages selected by the LLM

    for item in selected_links:
        page_type = item.get("type")
        page_url = item.get("url")

        print(f"Scraping {page_type}: {page_url}")
        combined_text += f"=== {page_type.upper()} CONTENT ===\n"
        combined_text += fetch_website_content(page_url) + "\n\n"

    return combined_text

### Function 3: Brochure Generator

In [24]:
def stream_brochure(combined_text, model_selection):
    """
    Sends the website text to the LLM and yields the final 
    brochure copy chunk-by-chunk in real-time.
    """
    # 1. System instructions for the copywriting style
    brochure_system_prompt = """
    You are a professional copywriter. Write an engaging sales and recruitment brochure using the text provided.
    Structure the brochure into these sections:
    1. Company Overview (Mission & Identity)
    2. Key Services / Products
    3. Careers & Culture (Why work here)
    4. Contact Info
    Format with clean Markdown headers and bullet points.
    """
    
    # 2. Start a streaming request
    stream = client.chat.completions.create(
        model=model_selection,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": f"Website Text Content:\n\n{combined_text}"}
        ],
        stream=True
    )
    
    brochure_text = ""
    # 3. Read chunks and yield them as they arrive
    for chunk in stream:
        chunk_content = chunk.choices[0].delta.content
        if chunk_content is not None:
            brochure_text += chunk_content
            yield brochure_text # yield pushes the current text state immediately

### Main function - Orchestrator & Gradio UI

In [25]:
def make_brochure(url, model_selection):
    # Step A: Filter links
    yield "Step 1: Discovering and filtering relevant pages..."
    relevant_links = filter_relevant_links(url, model_selection)
    
    if not relevant_links:
        yield "Error: Could not find any relevant links to scrape."
        return
        
    # Step B: Scrape content of selected pages
    yield f"Step 2: Scraping contents of {len(relevant_links)} sub-pages..."
    combined_text = scrape_all_data(url, relevant_links)
    
    # Step C: Stream brochure generation
    yield "Step 3: Writing brochure (streaming)..."
    # We yield the text chunks generated by our streaming helper function
    for text in stream_brochure(combined_text, model_selection):
        yield text

In [28]:
with gr.Blocks() as demo:
    # Title and description
    gr.Markdown("# <center>🏢 Local AI Brochure Generator</center>")
    gr.Markdown("<center>Enter a company homepage URL and let local LLMs crawl, parse, and compile a marketing brochure.</center>")
    
    # Row 1: Centered Inputs
    # We use 3 columns: empty spacing column on left (scale=1), 
    # inputs in center (scale=2), empty spacing column on right (scale=1)
    with gr.Row():

        with gr.Column(scale=2): # Inputs
            url_input = gr.Textbox(label="Company Website URL", value="https://www.google.com/")
            model_input = gr.Dropdown(
                label="Select Model", 
                choices=["gemma3:4b", "llama3.2:1b", "gemma3:270m"], 
                value="llama3.2:1b"
            )
            submit_btn = gr.Button("Generate Brochure", variant="primary")
        
    # Divider line
    gr.HTML("<hr>")
    
    # Row 2: Outputs (Placed Below Inputs)
    with gr.Row():
        
        with gr.Column(scale=3): # Wide output column
            output_panel = gr.Markdown(label="Generated Brochure")
        
    # 3. Connect button click event to our orchestrator function
    submit_btn.click(
        fn=make_brochure,
        inputs=[url_input, model_input],
        outputs=output_panel
    )
# Launch the app!
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
